# Doppler Geolocation

Recover a ground site's position from a single satellite pass of noisy, unscaled Doppler (range rate) measurements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skyfield.api import EarthSatellite, load

from thistle.utils import read_tle
from thistle.ground_sites import generate_range
from thistle.tracking import geolocate_doppler
from thistle.orbit_data import generate_lla

ts = load.timescale()
tles = read_tle("../tests/data/25544.tle")
sat = EarthSatellite(tles[0][0], tles[0][1], ts=ts)

## Generate synthetic Doppler

Pick a ground site near the ISS ground track, compute true range rate over a 10-minute pass window, then scale and add noise to simulate raw Doppler measurements. Only samples where the satellite is above 5° elevation are kept, mimicking a realistic ground station visibility constraint.

In [ ]:
from skyfield.api import wgs84
from thistle.utils import jday_datetime64

# True ground site: Boulder, CO
TRUE_LAT, TRUE_LON = 40.01, -105.27
MIN_ELEV = 5.0  # degrees

# Oblique pass (~23 deg peak elevation) for clear left/right ambiguity
t_center = np.datetime64("1998-11-21T23:22:20")
all_times = t_center + np.arange(-300, 300, 10, dtype="timedelta64[s]")

# Compute elevation to mask below-horizon samples
topos = wgs84.latlon(TRUE_LAT, TRUE_LON)
jd, fr = jday_datetime64(all_times)
t_sky = ts.tt_jd(jd, fr)
alt_deg, _, _ = (sat - topos).at(t_sky).altaz()
mask = alt_deg.degrees >= MIN_ELEV

times = all_times[mask]
minutes = (times - t_center) / np.timedelta64(1, "m")

# True range rate (only visible portion)
truth = generate_range(times, sat, sites=[(TRUE_LAT, TRUE_LON)])
true_rr = truth["range_rate_0"]

# Simulate unscaled Doppler: arbitrary scale + Gaussian noise
SCALE = 0.0073  # e.g. Hz per m/s
rng = np.random.default_rng(12)
noise_std = 0.5  # doppler units
doppler = SCALE * true_rr + rng.normal(0, noise_std, len(true_rr))

print(f"True site: Boulder, CO ({TRUE_LAT}, {TRUE_LON})")
print(f"Elevation mask: >= {MIN_ELEV} deg")
print(f"Scale: {SCALE},  Noise std: {noise_std}")
print(f"{len(times)} of {len(all_times)} samples above {MIN_ELEV} deg")

## Solve for ground site

In [3]:
solutions = geolocate_doppler(times, sat, doppler)

for i, sol in enumerate(solutions):
    label = "Best" if i == 0 else "Ambiguous"
    print(f"--- {label} solution ---")
    print(f"  Site:      ({sol.lat:.4f}, {sol.lon:.4f})")
    print(f"  Scale:     {sol.scale:.6f}")
    print(f"  RMS:       {sol.rms:.4f}")
    print(f"  Converged: {sol.converged}")
    print()

print(f"True site:   ({TRUE_LAT:.4f}, {TRUE_LON:.4f})")
print(f"Lat error:   {solutions[0].lat - TRUE_LAT:+.4f}\u00b0")
print(f"Lon error:   {solutions[0].lon - TRUE_LON:+.4f}\u00b0")

--- Best solution ---
  Site:      (34.3770, -111.3723)
  Scale:     0.007248
  RMS:       0.3836
  Converged: True

--- Ambiguous solution ---
  Site:      (39.9139, -105.3932)
  Scale:     0.007213
  RMS:       0.3838
  Converged: True

True site:   (40.0100, -105.2700)
Lat error:   -5.6330°
Lon error:   -6.1023°


## Plot: Doppler curve fit

In [ ]:
best = solutions[0]
ambig = solutions[1]

# Reconstruct fitted models from both solutions
best_fitted_rr = generate_range(times, sat, sites=[(best.lat, best.lon)])["range_rate_0"]
best_fitted_doppler = best.scale * best_fitted_rr

ambig_fitted_rr = generate_range(times, sat, sites=[(ambig.lat, ambig.lon)])["range_rate_0"]
ambig_fitted_doppler = ambig.scale * ambig_fitted_rr

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 5),
                                gridspec_kw={"height_ratios": [3, 1]})

ax1.scatter(minutes, doppler, s=8, alpha=0.6, label="Noisy Doppler")
ax1.plot(minutes, best_fitted_doppler, "r-", linewidth=2, label=f"Best fit (RMS {best.rms:.2f})")
ax1.plot(minutes, ambig_fitted_doppler, "m--", linewidth=1.5, alpha=0.7, label=f"Ambiguous fit (RMS {ambig.rms:.2f})")
ax1.plot(minutes, SCALE * true_rr, "k--", linewidth=1, alpha=0.5, label="Truth")
ax1.set_ylabel("Doppler (arb. units)")
ax1.legend()
ax1.set_title("Doppler Geolocation")

ax2.scatter(minutes, best.residuals, s=8, alpha=0.6, color="C2", label="Best")
ax2.scatter(minutes, ambig.residuals, s=8, alpha=0.4, color="m", label="Ambiguous")
ax2.axhline(0, color="k", linewidth=0.5)
ax2.set_ylabel("Residuals")
ax2.set_xlabel("Time from TCA (min)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot: Both solutions on ground track

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from thistle.ground_sites import visibility_circle

# Ground track over the pass
track_times = t_center + np.arange(-600, 600, 5, dtype="timedelta64[s]")
lla = generate_lla(track_times, sat)

# Visibility ring at 5 deg elevation using satellite altitude at TCA
tca_lla = generate_lla(np.array([t_center]), sat)
sat_alt_m = float(tca_lla["alt"][0])
vis_lats, vis_lons = visibility_circle(TRUE_LAT, TRUE_LON, 0.0, sat_alt_m, MIN_ELEV)
# Close the polygon
vis_lats = np.append(vis_lats, vis_lats[0])
vis_lons = np.append(vis_lons, vis_lons[0])

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()},
                        figsize=(10, 6))
ax.set_extent([-125, -90, 25, 50], crs=ccrs.PlateCarree())
ax.coastlines(linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.add_feature(cfeature.STATES, linewidth=0.2)
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

ax.plot(vis_lons, vis_lats, "g-", linewidth=1, alpha=0.7,
        label=f"{MIN_ELEV:.0f}\u00b0 visibility ring ({sat_alt_m/1e3:.0f} km)",
        transform=ccrs.Geodetic())
ax.plot(lla["lon"], lla["lat"], "b-", linewidth=1.5, alpha=0.5,
        label="Ground track", transform=ccrs.Geodetic())
ax.plot(TRUE_LON, TRUE_LAT, "k+", markersize=14, markeredgewidth=2,
        label=f"True ({TRUE_LAT:.2f}, {TRUE_LON:.2f})",
        transform=ccrs.PlateCarree())
ax.plot(best.lon, best.lat, "ro", markersize=8,
        label=f"Best (RMS {best.rms:.2f})",
        transform=ccrs.PlateCarree())
ax.plot(ambig.lon, ambig.lat, "rs", markersize=8, alpha=0.5,
        label=f"Ambiguous (RMS {ambig.rms:.2f})",
        transform=ccrs.PlateCarree())

ax.legend(loc="lower right")
ax.set_title("Doppler Geolocation \u2013 Boulder, CO")
plt.tight_layout()
plt.show()